# FraudShield Advanced: Inference Patterns Lab

## Scenario

FraudShield Risk Analytics needs to serve fraud predictions across multiple
channels with different latency, throughput, and cost requirements. As the MLOps
engineer, you will deploy the trained FraudShield model using three distinct
inference patterns and compare their characteristics:

1. **Serverless Inference** -- for sporadic, low-volume traffic with cost
   efficiency (scales to zero, pay per invocation)
2. **Batch Transform** -- for bulk offline scoring of historical transactions
   (ephemeral compute, no persistent endpoint)
3. **Multi-Model Endpoint** -- for serving multiple regional fraud models
   behind a single endpoint (shared infrastructure, LRU caching)

By the end of this lab, you will have hands-on experience with each pattern
and be able to articulate when to use each based on the Inference Decision
Matrix from the readings.

---

*This notebook runs on ml.t3.medium in SageMaker Studio. Inference resources
use ml.m5.xlarge (Free Tier eligible).*

In [ ]:
%pip install -q sagemaker boto3 pandas

In [ ]:
import sagemaker
import boto3
import json
import time
from datetime import datetime
from sagemaker.model import Model
from sagemaker.serverless import ServerlessInferenceConfig
from sagemaker.multidatamodel import MultiDataModel
from sagemaker.transformer import Transformer
from sagemaker import image_uris

sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
sm_runtime = boto3.client("sagemaker-runtime")
s3_client = boto3.client("s3")
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-inference-lab"
TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")

MODEL_ARTIFACT = f"s3://{default_bucket}/{PREFIX}/models/model.tar.gz"
CONTAINER_IMAGE = image_uris.retrieve("xgboost", region, version="1.5-1")

test_payload = "0.5,1.0,0.3,0.7,0.2,0.9,0.1,0.4,0.6,0.8"

print(f"Region          : {region}")
print(f"Role            : {role.split('/')[-1]}")
print(f"Default bucket  : {default_bucket}")
print(f"Container image : {CONTAINER_IMAGE}")
print(f"Model artifact  : {MODEL_ARTIFACT}")
print()
print("Update MODEL_ARTIFACT above if your trained model is at a different S3 path.")

---

## Concept Connection: Choosing the Right Pattern

Before deploying, review the **Inference Decision Matrix** from the readings.
The five decision criteria are:

1. **Latency requirement** -- sub-second, seconds, or minutes?
2. **Traffic pattern** -- steady, bursty, sporadic, or one-time?
3. **Payload size** -- kilobytes, megabytes, or gigabytes?
4. **Processing time** -- milliseconds, seconds, or minutes?
5. **Cost sensitivity** -- minimize cost or maximize performance?

The decision flowchart:

- Q1: Is this a one-time bulk scoring job? --> Batch Transform
- Q2: Payload > 6 MB or processing > 60 seconds? --> Async Inference
- Q3: Sub-second response required? --> Real-time Endpoint
- Q4: Sporadic, low-volume traffic? --> Serverless Inference

For each deployment pattern in this lab, consider how it maps to these criteria.
You will be asked to explain your reasoning at each Presentation Checkpoint.

---

## Step 1: Deploy Serverless Endpoint

Deploy the FraudShield model as a Serverless endpoint. Serverless scales to
zero when idle, so you pay nothing during quiet periods. You configure two
parameters:

- **MemorySizeInMB** (1024-6144, in 1024 MB increments): RAM for each
  inference container. Larger memory supports bigger models.
- **MaxConcurrency** (1-200): Maximum simultaneous requests.

The trade-off is cold start latency on the first request after idle time
(typically 5-30 seconds for model download and initialization).

In [ ]:
serverless_model_name = f"fs-lab-serverless-{TIMESTAMP}"

serverless_model = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    name=serverless_model_name,
)

print(f"Model object created: {serverless_model_name}")

In [ ]:
serverless_endpoint_name = f"fs-lab-serverless-ep-{TIMESTAMP}"

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5,
)

serverless_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name=serverless_endpoint_name,
)

print(f"Serverless endpoint deployed: {serverless_endpoint_name}")
print(f"  Memory: 2048 MB | Max concurrency: 5")

In [ ]:
print("Call 1 -- Cold start (container provisioned on demand):")
start = time.time()
response = sm_runtime.invoke_endpoint(
    EndpointName=serverless_endpoint_name,
    ContentType="text/csv",
    Body=test_payload,
)
cold_latency = time.time() - start
result = response["Body"].read().decode("utf-8")
print(f"  Prediction: {result}")
print(f"  Latency   : {cold_latency:.2f} seconds")

In [ ]:
print("Call 2 -- Warm (container already running):")
start = time.time()
response = sm_runtime.invoke_endpoint(
    EndpointName=serverless_endpoint_name,
    ContentType="text/csv",
    Body=test_payload,
)
warm_latency = time.time() - start
result = response["Body"].read().decode("utf-8")
print(f"  Prediction: {result}")
print(f"  Latency   : {warm_latency:.2f} seconds")

print(f"\nLatency comparison:")
print(f"  Cold start : {cold_latency:.2f}s")
print(f"  Warm       : {warm_latency:.2f}s")
print(f"  Difference : {cold_latency - warm_latency:.2f}s")

### Presentation Checkpoint

Be prepared to explain:

- What caused the latency difference between the cold and warm calls?
- What happens behind the scenes during a cold start (instance provisioning,
  model download from S3, model deserialization into memory)?
- For FraudShield, when would the cold start trade-off be acceptable?
  (Example: internal compliance scoring tool with < 100 requests/day)
- When would it be unacceptable?
  (Example: real-time card swipe scoring requiring sub-second response)
- What mitigation strategies exist? (Provisioned concurrency, warm-up
  requests, model size optimization)

In [ ]:
print("Deleting serverless endpoint and model...")

sm_client.delete_endpoint(EndpointName=serverless_endpoint_name)
print(f"  Deleted endpoint: {serverless_endpoint_name}")

try:
    sm_client.delete_endpoint_config(EndpointConfigName=serverless_endpoint_name)
    print(f"  Deleted endpoint config: {serverless_endpoint_name}")
except sm_client.exceptions.ClientError:
    pass

sm_client.delete_model(ModelName=serverless_model_name)
print(f"  Deleted model: {serverless_model_name}")
print("Serverless resources cleaned up.")

---

## Step 2: Run Batch Transform

Batch Transform is the right choice when you need to score an entire dataset
without maintaining a persistent endpoint. SageMaker provisions an ephemeral
cluster, processes all records, writes predictions to S3, and tears down the
cluster automatically. You pay only for the job duration.

Key configuration:
- **Instance type/count:** Determines processing speed and cost
- **Split type:** `Line` (each line is a record) or `None` (each file is one record)
- **Output path:** S3 location for prediction results
- **Join source:** Set to `Input` to append predictions to original records

In [ ]:
batch_data_key = f"{PREFIX}/batch-input/transactions_batch.csv"
batch_rows = []
for i in range(50):
    row = ",".join([f"{(i * 0.02 + j * 0.1) % 1:.2f}" for j in range(10)])
    batch_rows.append(row)

s3_client.put_object(
    Bucket=default_bucket,
    Key=batch_data_key,
    Body="\n".join(batch_rows).encode("utf-8"),
)

batch_input_s3 = f"s3://{default_bucket}/{batch_data_key}"
batch_output_s3 = f"s3://{default_bucket}/{PREFIX}/batch-output/"

print(f"Uploaded {len(batch_rows)} synthetic transaction records")
print(f"Input : {batch_input_s3}")
print(f"Output: {batch_output_s3}")

In [ ]:
batch_model_name = f"fs-lab-batch-{TIMESTAMP}"

batch_model = Model(
    model_data=MODEL_ARTIFACT,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
    name=batch_model_name,
)
batch_model.create(instance_type="ml.m5.xlarge")

transformer = Transformer(
    model_name=batch_model_name,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=batch_output_s3,
    sagemaker_session=sm_session,
    accept="text/csv",
)

print(f"Model: {batch_model_name}")
print(f"Transformer created (ml.m5.xlarge, 1 instance)")

In [ ]:
print("Starting batch transform job...")
transform_start = time.time()

transformer.transform(
    data=batch_input_s3,
    content_type="text/csv",
    split_type="Line",
)
transformer.wait()

transform_duration = time.time() - transform_start
print(f"Batch transform complete in {transform_duration:.0f} seconds.")
print(f"Output written to: {batch_output_s3}")

output_key = f"{PREFIX}/batch-output/transactions_batch.csv.out"
try:
    result_obj = s3_client.get_object(Bucket=default_bucket, Key=output_key)
    predictions = result_obj["Body"].read().decode("utf-8").strip().split("\n")
    print(f"\nTotal predictions: {len(predictions)}")
    print("First 5 predictions:")
    for i, pred in enumerate(predictions[:5]):
        print(f"  Record {i}: {pred}")
except Exception as e:
    print(f"Error reading results: {e}")

sm_client.delete_model(ModelName=batch_model_name)
print(f"\nCleaned up batch model: {batch_model_name}")
print("No endpoint to delete -- Batch Transform uses ephemeral compute.")

### Presentation Checkpoint

Be prepared to explain:

- How does Batch Transform differ from deploying an endpoint? (Ephemeral
  compute vs. persistent infrastructure, automatic cluster teardown, no
  scaling configuration needed)
- When would you choose Batch Transform over Serverless for FraudShield?
  (Nightly scoring of all historical transactions, no latency requirement,
  pay only for job duration)
- When would Serverless be better than Batch? (Individual requests arriving
  sporadically throughout the day, responses needed in seconds)
- What does `split_type='Line'` do? (Each line in the CSV is treated as a
  separate inference record)
- What would `join_source='Input'` add? (Appends predictions to original
  input rows for easy matching back to transaction IDs)

---

## Step 3: Deploy Multi-Model Endpoint

FraudShield operates in multiple regions (US-East, US-West, EU), each with
its own trained fraud model. Instead of deploying three separate real-time
endpoints (~$510/month total), a Multi-Model Endpoint hosts all three models
behind a single endpoint on shared infrastructure.

Key MME concepts:
- Model artifacts are stored under a common S3 prefix
- The client specifies which model via the `TargetModel` parameter
- Models are loaded into memory on demand and cached using LRU eviction
- All models must use the same inference container
- Adding a model = uploading a .tar.gz to S3 (no redeployment)

In [ ]:
mme_s3_prefix = f"s3://{default_bucket}/{PREFIX}/mme-models/"
mme_model_name = f"fs-lab-mme-{TIMESTAMP}"
mme_endpoint_name = f"fs-lab-mme-ep-{TIMESTAMP}"

model_artifacts = ["model_us_east.tar.gz", "model_us_west.tar.gz", "model_eu.tar.gz"]

source_bucket = MODEL_ARTIFACT.split("/")[2]
source_key = "/".join(MODEL_ARTIFACT.split("/")[3:])

for artifact in model_artifacts:
    dest_key = f"{PREFIX}/mme-models/{artifact}"
    s3_client.copy_object(
        CopySource={"Bucket": source_bucket, "Key": source_key},
        Bucket=default_bucket,
        Key=dest_key,
    )
    print(f"Uploaded: s3://{default_bucket}/{dest_key}")

print(f"\n3 regional model artifacts staged at: {mme_s3_prefix}")

In [ ]:
mme_model = MultiDataModel(
    name=mme_model_name,
    model_data_prefix=mme_s3_prefix,
    image_uri=CONTAINER_IMAGE,
    role=role,
    sagemaker_session=sm_session,
)

mme_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=mme_endpoint_name,
)

print(f"Multi-Model Endpoint deployed: {mme_endpoint_name}")
print(f"  Instance type : ml.m5.xlarge")
print(f"  Models        : {model_artifacts}")

In [ ]:
latencies = {}

for target_model in model_artifacts:
    print(f"Invoking TargetModel='{target_model}'...")
    start = time.time()
    response = sm_runtime.invoke_endpoint(
        EndpointName=mme_endpoint_name,
        TargetModel=target_model,
        ContentType="text/csv",
        Body=test_payload,
    )
    latency = time.time() - start
    prediction = response["Body"].read().decode("utf-8")
    latencies[target_model] = latency
    print(f"  Prediction: {prediction}  |  Latency: {latency:.2f}s")

In [ ]:
print(f"Re-invoking '{model_artifacts[0]}' (should be cached / hot):")
start = time.time()
response = sm_runtime.invoke_endpoint(
    EndpointName=mme_endpoint_name,
    TargetModel=model_artifacts[0],
    ContentType="text/csv",
    Body=test_payload,
)
cached_latency = time.time() - start
prediction = response["Body"].read().decode("utf-8")
print(f"  Prediction: {prediction}  |  Latency: {cached_latency:.2f}s")

print(f"\nLatency summary:")
for model, lat in latencies.items():
    print(f"  {model:25s}: {lat:.2f}s (cold -- first load)")
print(f"  {model_artifacts[0]:25s}: {cached_latency:.2f}s (hot -- cached in memory)")

### Presentation Checkpoint

Be prepared to explain:

- What is the difference between a cache hit (hot model) and a cache miss
  (cold model) on a Multi-Model Endpoint? What happens during each?
- How does the LRU eviction policy decide which models to keep in memory?
  If you have 100 models but memory for 20, which 20 stay loaded?
- Why must all models in an MME use the same inference container and
  instance type?
- How does adding a new model to an MME compare to creating a new
  single-model endpoint? (Upload .tar.gz to S3 prefix vs. create model +
  endpoint config + endpoint)
- What is the cost advantage of serving 3 regional models on one MME vs.
  3 separate real-time endpoints? (~$170/month vs. ~$510/month)

---

## MANDATORY Cleanup

All endpoints, endpoint configurations, and models created in this lab must
be deleted to avoid ongoing charges. The cells below remove every resource
and verify none remain. **Do not skip this step.**

In [ ]:
endpoints_to_delete = [
    serverless_endpoint_name,
    mme_endpoint_name,
]

models_to_delete = [
    serverless_model_name,
    batch_model_name,
    mme_model_name,
]

print("=" * 60)
print("CLEANUP: Deleting all inference resources")
print("=" * 60)

for ep_name in endpoints_to_delete:
    try:
        sm_client.delete_endpoint(EndpointName=ep_name)
        print(f"  Deleted endpoint: {ep_name}")
    except sm_client.exceptions.ClientError:
        print(f"  Endpoint already deleted or not found: {ep_name}")

    try:
        sm_client.delete_endpoint_config(EndpointConfigName=ep_name)
        print(f"  Deleted endpoint config: {ep_name}")
    except sm_client.exceptions.ClientError:
        print(f"  Endpoint config already deleted or not found: {ep_name}")

for model_name in models_to_delete:
    try:
        sm_client.delete_model(ModelName=model_name)
        print(f"  Deleted model: {model_name}")
    except sm_client.exceptions.ClientError:
        print(f"  Model already deleted or not found: {model_name}")

In [ ]:
print("=" * 60)
print("VERIFICATION: Checking for remaining resources")
print("=" * 60)

remaining_eps = sm_client.list_endpoints(
    NameContains="fs-lab",
    StatusEquals="InService",
)["Endpoints"]

if remaining_eps:
    print(f"\n  WARNING: {len(remaining_eps)} endpoint(s) still active:")
    for ep in remaining_eps:
        print(f"    - {ep['EndpointName']} ({ep['EndpointStatus']})")
    print("  Delete these manually before closing this session.")
else:
    print("  No active lab endpoints found.")

remaining_models = sm_client.list_models(
    NameContains="fs-lab",
    MaxResults=20,
)["Models"]

if remaining_models:
    print(f"\n  WARNING: {len(remaining_models)} model(s) still registered:")
    for m in remaining_models:
        print(f"    - {m['ModelName']}")
else:
    print("  No lab models found.")

print("\nCleanup complete. Lab finished.")